# Verify published services before lookup

Compare source evidence for the same linked branch and service. This notebook saves
the corroboration result, disagreements and the quotes used to make the decision.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Check the branch, service, publisher and date

A direct service statement from one publisher is a cited claim. The corroboration
check requires a second independent publisher with a compatible, explicit service
statement. Two pages from one publisher count together. Manually reviewed public
sources attach only when their name and location identify one branch in the snapshot.
Expired and conflicting statements remain visible. Check current availability by phone.

In [ ]:
import json

from facility_tool.examples import demo_reviewed_services
from facility_tool.reviewed_claims import attach_reviewed_claims
from facility_tool.verification import verify_services

claims = STORE.read("claims")
entries = demo_reviewed_services(config["as_of"]) if MODE == "demo" else json.loads(
    (ROOT / "non-ml/config/reviewed_service_evidence.json").read_text(encoding="utf-8")
)
reviewed, decisions = attach_reviewed_claims(
    entries, STORE.read("entities"), config["service_aliases"], config["as_of"],
)
combined = pd.concat([claims, reviewed], ignore_index=True) if not reviewed.empty else claims
verification = verify_services(combined, config["as_of"])
STORE.write("reviewed_service_links", decisions)
STORE.write("verified_claims", combined)
STORE.write("service_verification", verification)
show(decisions)
show(verification.groupby(["service", "verification_status"]).size().rename("branches").reset_index(), limit=60)